# 01 – Bee Haven: Datenbereinigung (N1)

**Ziel:** Die Sensordaten eines Standorts (`STADT`) aus `silver/processing` einheitlich und zuverlässig für die Analyse machen und als Zwischenstand für N2 speichern.

**Regeln (Pflichtenheft PH-ITS-002, Schritte 2.4–2.6 und 5.9a; Befund-Register Kap. 8.2):** R1 Zeitstempel → Datum/Uhrzeit · R2 Zeit → UTC · R3 fehlende Werte nur zählen · R4 flow-Duplikate behalten · **Gewicht einheitlich in Gramm** (Einheit der Rohdaten je Stadt aus `lakehouse/konfiguration/sensoren.csv`, Spalte `einheit_roh`)

**Grundsatz:** markieren statt löschen · `processing` wird nur gelesen, nie überschrieben · Standortregeln stehen in der Konfiguration, nicht im Code

**Ergebnistexte:** gelten für jeden Standort; die Zahlen stehen in der Ausgabe darüber. Die Tabellen „Kontrollwerte“ nennen die erwarteten Werte je Stadt (Stand 02.10.2026) – weicht ein neuer Lauf ab, haben sich Daten oder Regeln geändert.

## 0. Bibliotheken und Pfade

In [1]:
# --- Parameter: Standort (neue Stadt = Zeile in lakehouse/konfiguration/standorte.csv + Ordner in silver/processing) ---
STADT = "schwartau"                                       # Standort = Unterordner in bronze/silver/gold und Teil der Dateinamen

# --- Bibliotheken laden ---
import pandas as pd                      # pandas: Tabellen (DataFrames) einlesen, bereinigen, speichern
from pathlib import Path                 # Path: Ordner- und Dateipfade unabhängig vom Betriebssystem angeben
from datetime import datetime, timezone  # für den UTC-Zeitstempel im Dateinamen der Ausgabe

# --- Pfade festlegen (gleiche Struktur wie in Azure) ---
PROJEKT = Path.cwd().parent                               # Projektordner = eine Ebene über 03_notebooks_ITS-Data
SILVER = PROJEKT / "02_daten_ITS-Data" / "lakehouse" / "silver"    # lokaler Spiegel von lakehouse/silver
PROCESSING = SILVER / "processing" / STADT              # Eingang: unbereinigte Arbeitskopie (nur LESEN!)
CLEANED = SILVER / "cleaned" / STADT                    # Ausgang: bereinigte Dateien mit Zeitstempel

# --- Standortdaten aus der Konfiguration (Koordinaten, Zeitzone) ---
standorte = pd.read_csv(PROJEKT / "02_daten_ITS-Data" / "lakehouse" / "konfiguration" / "standorte.csv", index_col="stadt")
if STADT not in standorte.index:                          # unbekannte Stadt -> sofort abbrechen statt mit falschen Daten weiterrechnen
    raise ValueError(f"Standort '{STADT}' fehlt in konfiguration/standorte.csv – Zeile mit Koordinaten und Zeitzone ergänzen")
ORT = standorte.loc[STADT]                                # eine Zeile: name, breite, laenge, zeitzone, quelle

# --- Sensor-Regeln je Standort (Einheit der Rohdaten, gültiger Bereich) und bekannte Ausfallzeiträume ---
KONFIG = PROJEKT / "02_daten_ITS-Data" / "lakehouse" / "konfiguration"                  # Ordner mit den Konfigurationstabellen
sensoren_cfg = pd.read_csv(KONFIG / "sensoren.csv")                                    # eine Zeile je Stadt und Sensor
REGELN = sensoren_cfg[sensoren_cfg["stadt"] == STADT].set_index("sensor")              # Regeln dieser Stadt: einheit_roh, min, max
ausschluesse = pd.read_csv(KONFIG / "ausschluesse.csv")                                # bekannte Ausfallzeiträume (z. B. Stock nicht auf Waage)
AUSSCHLUESSE = ausschluesse[ausschluesse["stadt"] == STADT].copy()                     # nur Zeiträume dieser Stadt
AUSSCHLUESSE[["von_utc", "bis_utc"]] = AUSSCHLUESSE[["von_utc", "bis_utc"]].apply(pd.to_datetime, utc=True)  # Text -> UTC-Zeit
ZEITZONE = ORT["zeitzone"]                                # Ortszeit der Rohdaten (Deutschland: Europe/Berlin)

# --- Kontrolle: Welche Dateien liegen im Eingang? ---
for datei in sorted(PROCESSING.glob("*/*.csv")):          # alle CSV in den Sensor-Unterordnern suchen
    groesse_mb = datei.stat().st_size / 1_000_000         # Dateigröße von Byte in Megabyte umrechnen
    print(datei.relative_to(PROJEKT), f"{groesse_mb:.1f} MB")  # Pfad (ab Projektordner) und Größe ausgeben

02_daten_ITS-Data/lakehouse/silver/processing/schwartau/flow/flow_schwartau.csv 56.9 MB
02_daten_ITS-Data/lakehouse/silver/processing/schwartau/humidity/humidity_schwartau.csv 0.0 MB
02_daten_ITS-Data/lakehouse/silver/processing/schwartau/temperature/temperature_schwartau.csv 6.5 MB
02_daten_ITS-Data/lakehouse/silver/processing/schwartau/weight/weight_schwartau.csv 0.1 MB


**Ergebnis:** Alle 4 Dateien liegen im Eingang `processing/<stadt>/<sensor>/` (Ausgabe: Pfad und Größe).

| Kontrollwerte (MB) | flow | humidity | temperature | weight |
| :-- | --: | --: | --: | --: |
| Schwartau | 56,9 | 0,0 | 6,5 | 0,1 |
| Würzburg | 46,1 | 0,7 | 28,8 | 29,9 |

## 1. Daten einlesen (Original)

In [2]:
# --- Alle 4 CSV-Dateien einlesen ---
sensoren = ["flow", "humidity", "temperature", "weight"]  # Liste der Sensoren (= Ordnernamen in processing)
df = {}                                                   # leeres Dictionary: Sensorname -> Tabelle (DataFrame)
for s in sensoren:                                        # für jeden Sensor nacheinander ...
    pfad = PROCESSING / s / f"{s}_{STADT}.csv"          # ... den Dateipfad zusammensetzen
    df[s] = pd.read_csv(pfad)                             # ... die CSV als Tabelle einlesen (nur lesen!)
    print(f"{s:12} {len(df[s]):>10,} Zeilen   Spalten: {list(df[s].columns)}")  # Kontrolle ausgeben

flow          2,513,836 Zeilen   Spalten: ['timestamp', 'flow']
humidity          1,761 Zeilen   Spalten: ['timestamp', 'humidity']
temperature     253,430 Zeilen   Spalten: ['timestamp', 'temperature']
weight            1,761 Zeilen   Spalten: ['timestamp', 'weight']


**Ergebnis:** Je Sensor 2 Spalten (`timestamp` + Messwert); die Zeilenzahl zeigt den Messtakt (viele Zeilen = kurzer Takt).

| Kontrollwerte (Zeilen) | flow | humidity | temperature | weight |
| :-- | --: | --: | --: | --: |
| Schwartau | 2.513.836 | 1.761 | 253.430 | 1.761 |
| Würzburg | 2.055.520 | 20.845 | 958.831 | 1.035.861 |

## 2. Arbeitskopie erstellen

In [3]:
# --- Arbeitskopien anlegen: Die Originale in df bleiben zum Vergleich unverändert ---
cl = {s: tabelle.copy() for s, tabelle in df.items()}  # cl = "cleaned" (Namenskonvention aus der Kursvorlage)
print("Arbeitskopien:", list(cl.keys()))               # Kontrolle: Für welche Sensoren gibt es eine Kopie?

# --- Einheit vereinheitlichen: Gewicht intern immer in Gramm (Schwartau liefert g, Würzburg kg; Quelle: konfiguration/sensoren.csv) ---
FAKTOR_ZU_GRAMM = {"g": 1, "kg": 1000}                                    # Umrechnungsfaktor je Einheit der Rohdaten
einheit = REGELN.loc["weight", "einheit_roh"]                              # Einheit der Rohdaten dieser Stadt
cl["weight"]["weight"] = cl["weight"]["weight"] * FAKTOR_ZU_GRAMM[einheit]  # z. B. 52,7 kg -> 52.700 g
print(f"weight: Rohdaten in {einheit} -> intern in g (Faktor {FAKTOR_ZU_GRAMM[einheit]})")  # Kontrolle

Arbeitskopien: ['flow', 'humidity', 'temperature', 'weight']
weight: Rohdaten in g -> intern in g (Faktor 1)


**Ergebnis:** Arbeitskopien für alle 4 Sensoren angelegt (`cl`); die Originale (`df`) bleiben zum Vergleich unverändert. Das Gewicht wird nach `sensoren.csv` in Gramm umgerechnet (Ausgabe: Einheit und Faktor) – Schwartau liefert Gramm (Faktor 1), Würzburg Kilogramm (Faktor 1000).

## 3. Exploration der Daten

In [4]:
# --- Überblick je Sensor: Beispielzeilen, Datentypen, fehlende Werte, Wertebereich ---
for s in sensoren:                                         # für jeden Sensor nacheinander ...
    print(f"===== {s} =====")                              # ... Überschrift ausgeben
    print(cl[s].head(3))                                   # ... die ersten 3 Zeilen anzeigen
    print("Datentypen:", dict(cl[s].dtypes.astype(str)))   # ... Datentyp je Spalte (timestamp ist noch Text!)
    print("Fehlende Werte:", dict(cl[s].isna().sum()))     # ... Anzahl leerer Zellen je Spalte zählen
    print(cl[s][s].describe().round(2).to_dict())          # ... Min, Max, Mittelwert usw. der Messwert-Spalte
    print()                                                # ... Leerzeile für bessere Lesbarkeit

===== flow =====
             timestamp  flow
0  2017-01-01 14:15:00     0
1  2017-01-01 14:16:00     0
2  2017-01-01 14:17:00     0
Datentypen: {'timestamp': 'str', 'flow': 'int64'}
Fehlende Werte: {'timestamp': np.int64(0), 'flow': np.int64(0)}
{'count': 2513836.0, 'mean': 1.62, 'std': 54.33, 'min': -631.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 7999.0}

===== humidity =====
             timestamp  humidity
0  2017-01-01 13:00:00     98.04
1  2017-01-02 01:00:00     98.61
2  2017-01-02 13:00:00     99.00
Datentypen: {'timestamp': 'str', 'humidity': 'float64'}
Fehlende Werte: {'timestamp': np.int64(0), 'humidity': np.int64(12)}
{'count': 1749.0, 'mean': 83.99, 'std': 26.5, 'min': -100.0, '25%': 79.33, '50%': 91.07, '75%': 96.61, 'max': 100.0}

===== temperature =====
             timestamp  temperature
0  2017-01-01 14:10:00          NaN
1  2017-01-01 14:15:00        12.34
2  2017-01-01 14:20:00        12.27
Datentypen: {'timestamp': 'str', 'temperature': 'float64'}
Fehlende Werte

**Ergebnis:** Die Exploration bestätigt die Befunde aus dem Pflichtenheft: `timestamp` ist Text (R1), es gibt fehlende Werte (R3), und die Minima zeigen Fehlerwerte, die N2 prüft – flow negativ = Ausflüge (R8), humidity unter 0 % (R6), Temperatur- und Gewichtsausreißer (R5, R7).

| Kontrollwerte (Minimum) | flow | humidity | temperature | weight |
| :-- | --: | --: | --: | --: |
| Schwartau | −631 | −100 | −7,26 | −172,44 g |
| Würzburg | −674 | siehe Ausgabe | bis −227 °C (Fehlerwerte) | siehe Ausgabe (kg) |

## 4. Datentypen und Zeitzone (R1, R2)
**R1:** `timestamp` ist Text → in Datum/Uhrzeit umwandeln, sonst kann man nicht sortieren, gruppieren oder rechnen.

**R2:** Die Zeiten sind deutsche Ortszeit mit Sommerzeit. Im Oktober gibt es 02:00–03:00 Uhr doppelt → Umrechnung in **UTC** (ohne Sommerzeit, eindeutig).

Die Originalspalte `timestamp` bleibt erhalten (markieren statt löschen); neu ist die Spalte `zeit_utc`.

In [5]:
# --- Funktion: deutsche Ortszeit (Text) -> eindeutige UTC-Zeit ---
def zu_utc(zeit_text):
    """Wandelt Zeitstempel-Text in deutscher Ortszeit (mit Sommerzeit) in UTC um."""
    zeit = pd.to_datetime(zeit_text, format="%Y-%m-%d %H:%M:%S")  # R1: Text -> Datum/Uhrzeit (Format fest vorgegeben)
    neustart = (zeit.diff() < pd.Timedelta(hours=-2)).cumsum()  # Zeit springt weit zurück = neue Messreihe (flow hat 2)
    wiederholung = zeit.groupby([neustart, zeit]).cumcount()    # 0 = Uhrzeit kommt zum 1. Mal vor, 1 = zum 2. Mal
    sommerzeit = (wiederholung == 0).to_numpy()                 # 1. Vorkommen = noch Sommerzeit, 2. = schon Winterzeit
    zeit = zeit.dt.tz_localize(ZEITZONE,                        # R2: Zeitzone des Standorts zuordnen (Standort-Tabelle) ...
                               ambiguous=sommerzeit,            # ... doppelte Oktober-Stunde nach der Regel oben auflösen
                               nonexistent="raise")             # ... Fehler melden, falls eine Zeit in der März-Lücke liegt
    return zeit.dt.tz_convert("UTC")                            # ... und in UTC umrechnen (keine Sommerzeit)

In [6]:
# --- R1 + R2 auf alle Sensoren anwenden ---
for s in sensoren:                                           # für jeden Sensor ...
    cl[s]["zeit_utc"] = zu_utc(cl[s]["timestamp"])           # ... neue Spalte zeit_utc (Original timestamp bleibt!)
    print(f"{s:12} erste Zeit: {cl[s]['zeit_utc'].iloc[0]}   Typ: {cl[s]['zeit_utc'].dtype}")  # Kontrolle

# --- Beweis: humidity misst immer zur gleichen UTC-Uhrzeit (lokal: Sommer 02/14 Uhr, Winter 01/13 Uhr) ---
print("humidity UTC-Uhrzeiten:", sorted(cl["humidity"]["zeit_utc"].dt.strftime("%H:%M").unique()))

flow         erste Zeit: 2017-01-01 13:15:00+00:00   Typ: datetime64[us, UTC]
humidity     erste Zeit: 2017-01-01 12:00:00+00:00   Typ: datetime64[us, UTC]
temperature  erste Zeit: 2017-01-01 13:10:00+00:00   Typ: datetime64[us, UTC]
weight       erste Zeit: 2017-01-01 12:00:00+00:00   Typ: datetime64[us, UTC]
humidity UTC-Uhrzeiten: ['00:00', '12:00']


**Ergebnis:** Alle 4 Sensoren haben die neue Spalte `zeit_utc` vom Typ `datetime64[us, UTC]`.

Prüfung der Zeitzone: Misst ein Sensor in festem Takt (z. B. humidity in Schwartau 00:00 und 12:00 UTC, lokal im Sommer 02/14 Uhr, im Winter 01/13 Uhr), liegen die UTC-Uhrzeiten nach der Umrechnung immer gleich → Beleg, dass die Rohdaten deutsche Ortszeit mit Sommerzeit sind (Pflichtenheft E2). Würzburg misst humidity stündlich, daher erscheinen alle 24 Stunden.

## 5. Duplikate (R4)

Jetzt, **nach** der UTC-Umrechnung, lassen sich Duplikate richtig beurteilen (Befund E1).

Prüfung 1: Gibt es noch doppelte Zeiten innerhalb einer Messreihe? → erwartet: 0

Prüfung 2: Wie viele komplett gleiche Zeilen haben die Rohdaten? → bei flow sind das echte Minuten mit dem Wert 0 → **behalten**

In [7]:
# --- Duplikate prüfen (nach der UTC-Umrechnung) ---
for s in sensoren:                                                        # für jeden Sensor ...
    reihe = (cl[s]["zeit_utc"].diff() < pd.Timedelta(hours=-2)).cumsum()  # ... Messreihe bestimmen (flow: 0 und 1)
    doppelte_zeit = cl[s].assign(reihe=reihe).duplicated(subset=["reihe", "zeit_utc"]).sum()  # ... gleiche Zeit in derselben Reihe zählen
    gleiche_rohzeilen = df[s].duplicated().sum()                          # ... komplett gleiche Zeilen in den ROHdaten zählen
    print(f"{s:12} doppelte UTC-Zeiten: {doppelte_zeit:>3}   gleiche Rohzeilen: {gleiche_rohzeilen:>9,}")  # ... Ergebnis ausgeben


flow         doppelte UTC-Zeiten:   0   gleiche Rohzeilen:   697,348
humidity     doppelte UTC-Zeiten:   0   gleiche Rohzeilen:         0
temperature  doppelte UTC-Zeiten:   0   gleiche Rohzeilen:         0
weight       doppelte UTC-Zeiten:   0   gleiche Rohzeilen:         0


**Ergebnis:** Doppelte UTC-Zeiten müssen überall 0 sein – scheinbare Duplikate entstehen nur durch die Zeitumstellung (E1, E4). Gleiche Rohzeilen sind bei flow echte Minuten mit dem Wert 0 und werden behalten (R4); bei anderen Sensoren sind es gleiche Messwerte zu verschiedenen Zeiten im Rohformat und kein Fehler.

| Kontrollwerte (gleiche Rohzeilen) | flow | humidity | temperature | weight |
| :-- | --: | --: | --: | --: |
| Schwartau | 697.348 | 0 | 0 | 0 |
| Würzburg | 623.586 | 0 | 121.813 | 10 |

## 5a. Zeitabstände und versteckte Lücken
**Warum:** Eine Messreihe kann vollständig aussehen und trotzdem Lücken haben – dann fehlen Zeilen, statt dass NaN drinsteht. Das sieht man nur an den Zeitabständen.

**Eigene Erwartung:** Jeder Sensor hat einen normalen Takt (häufigster Abstand). Einzelne Lücken sind normal, längere Ausfälle möglich (z. B. Station außer Betrieb).

In [8]:
# --- Zeitabstände je Messreihe prüfen (nach UTC) ---
luecken_protokoll = []                                                  # Sammlung der Ergebnisse
for s in sensoren:                                                      # für jeden Sensor ...
    zeit = cl[s]["zeit_utc"]                                            # ... UTC-Zeitspalte
    reihe = (zeit.diff() < pd.Timedelta(hours=-2)).cumsum()             # ... Messreihe (flow: 0 = Ausflug, 1 = Einflug)
    abstand = zeit.groupby(reihe).diff().dropna()                       # ... Abstand zur vorherigen Zeile derselben Reihe
    normal = abstand.mode()[0]                                          # ... häufigster Abstand = normales Messintervall
    zu_gross = abstand[abstand > normal]                                # ... alle Abstände größer als normal = Lücken
    luecken_protokoll.append({"sensor": s, "normal": str(normal),       # ... Ergebnis festhalten:
                              "luecken": len(zu_gross),                 #     Anzahl Lücken
                              "davon_ueber_1_tag": int((zu_gross > pd.Timedelta(days=1)).sum()),
                              "groesste": str(zu_gross.max()) if len(zu_gross) else "-"})
pd.DataFrame(luecken_protokoll)                                         # Übersicht als Tabelle

,sensor,normal,luecken,davon_ueber_1_tag,groesste
0,flow,0 days 00:01:00,120,4,5 days 09:32:00
1,humidity,0 days 12:00:00,0,0,-
2,temperature,0 days 00:05:00,0,0,-
3,weight,0 days 12:00:00,0,0,-


**Ergebnis:** Die Tabelle zeigt je Sensor den normalen Takt, die Zahl der versteckten Lücken (Abstand größer als normal), davon über 1 Tag, und die größte Lücke. Diese Lücken stehen nicht als fehlende Werte in Kapitel 6 – man sieht sie nur hier.

| Kontrollwerte | flow | humidity | temperature | weight |
| :-- | :-- | :-- | :-- | :-- |
| Schwartau | 1 Min., 120 Lücken, 4 > 1 Tag, größte 5 Tage 09:32 | 12 Std., lückenlos | 5 Min., lückenlos | 12 Std., lückenlos |
| Würzburg | 1 Min., 340 Lücken, 6 > 1 Tag, größte 146 Tage 18:48 (Station Mai–Okt. 2018 außer Betrieb) | 1 Std., 1 Lücke | 5 Min., 208.421 kurze Unregelmäßigkeiten | 1 Min., 35 Lücken, größte 146 Tage |

→ Umgang mit Lücken: in Silver nichts auffüllen, kurze Lücken erst in Gold mit Kennzeichen (N4, Pflichtenheft P6).

## 6. Fehlende Werte (R3)

**Regel:** nur zählen und dokumentieren, **nichts auffüllen**.

**Warum?** Erfundene Werte (z. B. ein Mittelwert) täuschen Messungen vor, die es nie gab. Wie mit Lücken umgegangen wird, entscheidet erst Gold (N4) bzw. die Analyse (N5) – Entscheidung P6 = c: in Silver nichts auffüllen (Pflichtenheft Schritt 3.1).

In [9]:
# --- Fehlende Werte zählen (R3: nur zählen, nichts auffüllen) ---
protokoll = []                                                # leere Liste für das Bereinigungsprotokoll
for s in sensoren:                                            # für jeden Sensor ...
    fehlend = int(cl[s][s].isna().sum())                      # ... leere Messwerte zählen
    anteil = fehlend / len(cl[s]) * 100                       # ... Anteil in Prozent berechnen
    protokoll.append({"sensor": s,                            # ... eine Zeile ins Protokoll schreiben:
                      "zeilen": len(cl[s]),                   #     Anzahl Zeilen
                      "fehlend": fehlend,                     #     Anzahl fehlender Werte
                      "anteil_%": round(anteil, 2)})          #     Anteil in Prozent (2 Nachkommastellen)
pd.DataFrame(protokoll)                                       # Protokoll als Tabelle anzeigen

,sensor,zeilen,fehlend,anteil_%
0,flow,2513836,0,0.00
1,humidity,1761,12,0.68
2,temperature,253430,2032,0.80
3,weight,1761,12,0.68


**Ergebnis:** Fehlende Messwerte je Sensor mit Anteil – nicht aufgefüllt, nur dokumentiert (R3).

| Kontrollwerte (fehlend) | flow | humidity | temperature | weight |
| :-- | --: | --: | --: | --: |
| Schwartau | 0 | 12 (0,68 %) | 2.032 (0,80 %) | 12 (0,68 %) |
| Würzburg | 0 | 3.576 (17,16 %) | 164.580 (17,16 %) | 0 |

## 7. Speichern (Abschluss N1)

**Ziel:** Das Ergebnis von N1 als Zwischenstand für N2 (Datenqualität) sichern.

**Wohin?** `02_daten_ITS-Data/zwischenstand/` – nicht nach `processing` (wird nie überschrieben) und noch nicht nach `cleaned` (erst nach N2).

**Warum Parquet statt CSV?** Parquet behält die Datentypen (z. B. `zeit_utc` als UTC-Zeit), CSV macht alles wieder zu Text. Außerdem ca. halb so groß und schneller.

**Kontrollwert (Ablage laut Plan):** 4 Dateien `<sensor>_<stadt>_n1.parquet` + `n1_protokoll.csv`

In [10]:
# --- N1-Ergebnis speichern (Zwischenstand für N2) ---
ZWISCHENSTAND = PROJEKT / "02_daten_ITS-Data" / "zwischenstand"       # Ordner für Zwischenergebnisse zwischen den Notebooks
ZWISCHENSTAND.mkdir(parents=True, exist_ok=True)             # Ordner anlegen, falls er noch nicht existiert
for s in sensoren:                                           # für jeden Sensor ...
    ziel = ZWISCHENSTAND / f"{s}_{STADT}_n1.parquet"        # ... Dateiname festlegen
    cl[s].to_parquet(ziel, index=False)                      # ... als Parquet speichern (behält Datentypen inkl. UTC)
    groesse_mb = ziel.stat().st_size / 1_000_000             # ... Dateigröße in MB
    print(f"{s:12} gespeichert: {ziel.relative_to(PROJEKT)}  ({groesse_mb:.1f} MB)")  # ... Kontrolle ausgeben
pd.DataFrame(protokoll).to_csv(ZWISCHENSTAND / "n1_protokoll.csv", index=False)       # Protokoll (fehlende Werte) sichern
print("Protokoll gespeichert: n1_protokoll.csv")                                        # Kontrolle

flow         gespeichert: 02_daten_ITS-Data/zwischenstand/flow_schwartau_n1.parquet  (28.2 MB)
humidity     gespeichert: 02_daten_ITS-Data/zwischenstand/humidity_schwartau_n1.parquet  (0.0 MB)
temperature  gespeichert: 02_daten_ITS-Data/zwischenstand/temperature_schwartau_n1.parquet  (3.4 MB)
weight       gespeichert: 02_daten_ITS-Data/zwischenstand/weight_schwartau_n1.parquet  (0.0 MB)
Protokoll gespeichert: n1_protokoll.csv


**Ergebnis:** 4 Parquet-Dateien `<sensor>_<stadt>_n1.parquet` in `02_daten_ITS-Data/zwischenstand/` (Parquet ca. halb so groß wie CSV) und Protokoll `n1_protokoll.csv`. **N1 abgeschlossen** → weiter mit N2.

| Kontrollwerte (MB) | flow | humidity | temperature | weight |
| :-- | --: | --: | --: | --: |
| Schwartau | 28,2 | 0,0 | 3,4 | 0,0 |
| Würzburg | 22,7 | 0,4 | 12,5 | 12,1 |